In [1]:
import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.backtester.simulator import simulate
from qubx.core.basics import DataType, Signal
from qubx.core.interfaces import (
    IStrategy,
    IStrategyContext,
    IStrategyInitializer,
    ITimeProvider,
    MarketEvent,
    TriggerEvent,
)
from qubx.core.lookups import lookup
from qubx.data.cache import CachedStorage, MemoryCache
from qubx.data.guards import TimeGuardedStorage
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries


class FixedTimeProvider(ITimeProvider):
    """
    Test time provider returning a fixed time.
    """

    def __init__(self, time: str | np.datetime64):
        self._time = np.datetime64(time, "ns") if isinstance(time, str) else time

    def time(self) -> np.datetime64:
        return self._time

    def set_time(self, t: str):
        self._time = np.datetime64(t, "ns")


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


## Datareaders 

In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [3]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [4]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

(numpy.datetime64('2017-08-24T13:01:12'),
 numpy.datetime64('2017-08-24T13:01:59'))

In [5]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [6]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

[[2025-12-01T00:00:00.000000000]	90388.00000 (0.01) | 90389.00000 (0.01),
 [2025-12-01T00:00:01.000000000]	90380.00000 (0.01) | 90386.00000 (0.01),
 [2025-12-01T00:00:02.000000000]	90364.00000 (0.01) | 90370.00000 (0.01)]

In [7]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote", None, None).transform(TypedGenericSeries())

                         bid      ask  bid_size  ask_size
time                                                     
2025-12-01 00:00:00  90388.0  90389.0    0.0101    0.0113
2025-12-01 00:00:01  90380.0  90386.0    0.0101    0.0113
2025-12-01 00:00:02  90364.0  90370.0    0.0109    0.0113
2025-12-01 00:00:03  90366.0  90367.0    0.0050    0.0060
2025-12-01 00:00:04  90358.0  90361.0    0.0005    0.0236
...                      ...      ...       ...       ...
2025-12-01 23:59:55  86317.0  86318.0    0.0022    0.0071
2025-12-01 23:59:56  86317.0  86318.0    0.0022    0.0131
2025-12-01 23:59:57  86317.0  86318.0    0.0022    0.1869
2025-12-01 23:59:58  86291.0  86298.0    0.0641    0.0391
2025-12-01 23:59:59  86290.0  86291.0    0.0050    0.1792

[83631 rows x 4 columns]

In [8]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [9]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

-[MULTI DATA]-
 | BTCUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)
 | ETHUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)

## Time guarded 

In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
c_time = FixedTimeProvider("2021-01-01")
gstor = TimeGuardedStorage(stor, c_time)
gr = gstor.get_reader("BINANCE.UM", "SWAP")

c_time.set_time("2023-07-01 02:00")
gr.read("BTCUSDT", "ohlc(1h)", "2023-01-01", "2024-10-01")

BTCUSDT(ohlc(1h))[2023-06-01 00:00:00 : 2023-07-01 01:00:00] : (722 x 10)

In [3]:
stor2 = StorageRegistry.get("qdb::quantlab")

c_time2 = FixedTimeProvider("2021-01-01")
gstor2 = TimeGuardedStorage(stor2, c_time2)

gr2 = gstor2.get_reader("BINANCE.UM", "SWAP")

In [4]:
c_time2.set_time("2023-07-01 02:00")
gr2.read("BTCUSDT", "ohlc(1h)", "2023-01-01", "now")

BTCUSDT(ohlc(1h))[2023-01-01 00:00:00 : 2023-07-01 01:00:00] : (4346 x 11)

In [5]:
gr2.read("BTCUSDT", "funding_rate", "2023-01-01", "now")

BTCUSDT(funding_rate)[EMPTY] : (0 x 7)

In [6]:
gr2 = gstor2.get_reader("COINGECKO", "FUNDAMENTAL")
gr2.get_data_id()[-10:]

['ZORA', 'ZRC', 'ZRO', 'ZRX', 'ZTX', '小股东', '币安人生', '我踏马来了', '老子', '雪球']

In [15]:
c_time2.set_time("2022-01-01 23:59")
gr2.read([], "FUNDAMENTAL", "2021-12-01", "now").to_pd(True)

asset        metric         value
timestamp  symbol                                                      
2021-12-01 1000000BABYDOGE  1000000BABYDOGE    market_cap  4.289261e+08
           1000000BABYDOGE  1000000BABYDOGE         price  2.481952e-09
           1000000BABYDOGE  1000000BABYDOGE  total_volume  1.424207e+07
           1000000VINU          1000000VINU    market_cap  6.879704e+06
           1000000VINU          1000000VINU         price  4.134763e-08
...                                     ...           ...           ...
2022-01-01 ZIL                          ZIL         price  7.501285e-02
           ZIL                          ZIL  total_volume  9.449119e+07
           ZRX                          ZRX  total_volume  6.143750e+07
           ZRX                          ZRX    market_cap  6.862060e+08
           ZRX                          ZRX         price  8.068679e-01

[42963 rows x 3 columns]

## Cached Storage

In [ ]:
from qubx import QubxLogConfig

QubxLogConfig.set_log_level("DEBUG")

2026-02-20 09:46:31.333 [ 🐞 ] (questdb) Connected to QuestDB at quantlab:8812
2026-02-20 09:47:47.112 [ 🐞 ] (questdb) Connected to QuestDB at quantlab:8812
2026-02-20 09:47:47.113 [ 🐞 ] (questdb) Closing connection
2026-02-20 09:47:47.195 [ 🐞 ] (questdb) Bootstrapping manifest for table: coingecko.fundamental


In [3]:
# stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor = StorageRegistry.get("qdb::quantlab")
cached = CachedStorage(stor, cache_factory=lambda: MemoryCache(), prefetch_period="2W")

cr = cached.get_reader("BINANCE.UM", "SWAP")

cr.read("BTCUSDT", "ohlc(1h)", "2020-01-01", "now")

BTCUSDT(ohlc(1h))[2020-01-01 00:00:00 : 2026-02-18 23:00:00] : (53784 x 11)

In [10]:
print(cached._readers['BINANCE.UM:SWAP']._cache._ranges)

{'ohlc(1h)': [('2020-01-01 00:00:00', '2026-02-20 09:46:31.945741')]}


In [11]:
cr.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)", "2020-01-01", "2026-01-01")

-[MULTI DATA]-
 | BTCUSDT(ohlc(1h))[2020-01-01 00:00:00 : 2025-12-31 23:00:00] : (52608 x 11)
 | ETHUSDT(ohlc(1h))[2020-01-01 00:00:00 : 2025-12-31 23:00:00] : (52608 x 11)

In [12]:
stor = StorageRegistry.get("qdb::quantlab")
cached = CachedStorage(stor, cache_factory=lambda: MemoryCache(), prefetch_period="2W")

cr2 = cached.get_reader("COINGECKO", "FUNDAMENTAL")
xr = cached.get_reader("BINANCE.UM", "SWAP")

In [18]:
%%time
fdd = cr2.read([], "fundamental", "2022-01-01", "2023-01-01")

CPU times: user 84.9 ms, sys: 8.15 ms, total: 93.1 ms
Wall time: 92.4 ms


In [19]:
%%time
fdd2 = cr2.read([], "fundamental", "2022-01-01", "2023-02-20")

CPU times: user 47.6 ms, sys: 11.9 ms, total: 59.5 ms
Wall time: 58.9 ms


In [20]:
%%time
cr2.read([], "fundamental", "2023-01-01", "2023-02-01").to_pd(1);

CPU times: user 100 ms, sys: 61 μs, total: 100 ms
Wall time: 98.3 ms


In [22]:
xr.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)", "2022-01-01", "2023-02-01")

-[MULTI DATA]-
 | BTCUSDT(ohlc(1h))[2022-01-01 00:00:00 : 2023-01-31 23:00:00] : (9504 x 11)
 | ETHUSDT(ohlc(1h))[2022-01-01 00:00:00 : 2023-01-31 23:00:00] : (9504 x 11)

In [23]:
# stor = StorageRegistry.get("qdb::quantlab")
# cached = CachedStorage(stor, cache_factory=lambda: MemoryCache(), prefetch_period="2W")

# cr2 = cached.get_reader("COINGECKO", "FUNDAMENTAL")
fdd2 = cr2.read([], "fundamental", "2022-01-01", "2023-02-01")
fdd2.to_pd(True)

asset        metric         value
timestamp  symbol                                                      
2022-01-01 1000000BABYDOGE  1000000BABYDOGE    market_cap  3.213341e+08
           1000000BABYDOGE  1000000BABYDOGE         price  1.885529e-09
           1000000BABYDOGE  1000000BABYDOGE  total_volume  4.489650e+06
           1000000VINU          1000000VINU    market_cap  3.589154e+06
           1000000VINU          1000000VINU         price  1.990956e-08
...                                     ...           ...           ...
2023-01-31 ZIL                          ZIL    market_cap  4.303766e+08
           ZIL                          ZIL         price  2.683693e-02
           ZRX                          ZRX    market_cap  1.817993e+08
           ZRX                          ZRX  total_volume  1.530772e+07
           ZRX                          ZRX         price  2.150546e-01

[579195 rows x 3 columns]

In [24]:
from qubx.data.transformers import PandasFrame

In [ ]:
%%time
fdd2.transform(PandasFrame(True))
fdd2.to_pd(True)

CPU times: user 205 ms, sys: 16.1 ms, total: 221 ms
Wall time: 209 ms


asset        metric         value
timestamp  symbol                                                      
2022-01-01 1000000BABYDOGE  1000000BABYDOGE    market_cap  3.213341e+08
           1000000BABYDOGE  1000000BABYDOGE         price  1.885529e-09
           1000000BABYDOGE  1000000BABYDOGE  total_volume  4.489650e+06
           1000000VINU          1000000VINU    market_cap  3.589154e+06
           1000000VINU          1000000VINU         price  1.990956e-08
...                                     ...           ...           ...
2023-01-31 ZIL                          ZIL    market_cap  4.303766e+08
           ZIL                          ZIL         price  2.683693e-02
           ZRX                          ZRX    market_cap  1.817993e+08
           ZRX                          ZRX  total_volume  1.530772e+07
           ZRX                          ZRX         price  2.150546e-01

[579195 rows x 3 columns]

In [ ]:
fdd.to_pd(1) == fdd2.to_pd(1)

In [25]:
fd1 = cr2.read(['BTC', 'ETH'], "fundamental", "2020-01-01", "2026-01-01")
fd1

-[MULTI DATA]-
 | BTC(fundamental)[2021-11-27 00:00:00 : 2020-02-27 00:00:00] : (6576 x 4)
 | ETH(fundamental)[2025-04-22 00:00:00 : 2020-02-06 00:00:00] : (6576 x 4)

## Smoke test run

In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/multi/")
# stor = StorageRegistry.get("qdb::quantlab")
stor.get_exchanges()

['BINANCE.UM', 'KRAKEN.F', 'COINGECKO', 'HYPERLIQUID']

In [3]:
rr = stor.get_reader("BINANCE.UM", "SWAP")
print(rr.get_data_types("BTCUSDT"))
rr.read("BTCUSDT", "features", "2026-01-01", "now").to_records()[:4]

[quote, 'ohlc(1h)', 'features']


[[2026-01-01T01:00:00.000000000]	 {'f1': 1.0, 'f2': 11.0},
 [2026-01-01T02:00:00.000000000]	 {'f1': 2.0, 'f2': 21.0},
 [2026-01-01T03:00:00.000000000]	 {'f1': 3.0, 'f2': 31.0},
 [2026-01-01T04:00:00.000000000]	 {'f1': 4.0, 'f2': 41.0}]

In [4]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        initializer.set_event_schedule("1h -1s")

        # initializer.set_base_subscription("ohlc(1h)")
        initializer.set_base_subscription("ohlc_quotes(1h)")
        # initializer.subscribe("ohlc_trades(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("ohlc_quotes(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("features")
        # self._show = True

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        # qq = ''
        # for i in ctx.instruments:
        #     qq += f" | {i}: {ctx.quote(i).mid_price()}"
        # if data.type == "quote":

        # if self._show and data.type == "quote":
        #     self._show = False
        #     logger.info(data)

        # if data.type == "features":
            # logger.info(data)
        #     logger.info(data)
        #     self._show = True
        # logger.info(data)
        pass

    def on_event(self, ctx: IStrategyContext, event: TriggerEvent) -> list[Signal] | Signal | None:
        # logger.info(" | ".join([f"{i}: {ctx.quote(i).mid_price()}" for i in ctx.instruments]))
        d = ctx.ohlc(ctx.instruments[0], "1h", 10)
        # logger.info(f" === Time: <r>{ctx.time()}</r> ===\n<g>" + str(d.pd()[["open","high","low","close","volume"]]) + "</g>\n - - - - - - -")

        quotes = ctx.get_cached_market_data(ctx.instruments[0], "quote")
        # fund = ctx.(ctx.instruments[0], "quote")

        # logger.info("\n<g>" + str(d.pd()[["open","high","low","close","volume"]]) + "</g>\n - - - - - - -")
        # logger.info(quotes)

        aux = ctx.get_aux_reader("COINGECKO", "FUNDAMENTAL")
        logger.info(aux.read("__ALL__", "fundamental", "2020-01-01", "now"))

        # logger.info(aux.read(aux.get_data_id(), "fundamental", "2020-01-01", "now"))


In [5]:
stor0 = StorageRegistry.get("qdb::quantlab")

In [11]:
simulate(
    Test1(), 
    data=stor0, capital=1000, 
    aux_data=stor,
    # start="2023-06-01", stop="2023-06-02",
    start="2025-02-01 22:00", stop="2025-02-02 04:00",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        # "KRAKEN.F:SWAP:BTCUSD",
        # "HYPERLIQUID:SWAP:BTCUSDC",
    ], 
    debug="DEBUG"
);

2026-02-16 16:27:25.414 [ 🐞 ] (runner) [Simulator] :: Preparing simulated trading on ['BINANCE.UM'] for 1000 USDT
2026-02-16 16:27:25.415 [ ℹ️ ] (data) SimulatedDataProvider.BINANCE.UM is initialized
2026-02-16 16:27:25.415 [ ℹ️ ] (processing) Setting event schedule to 1h -1s
2026-02-16 16:27:25.416 [ 🐞 ] (context) [StrategyContext] :: Auto-assigned SimulationTransferManager
2026-02-16 16:27:25.417 [ 🐞 ] (runner) [SimulationRunner] :: Setting warmups: {'ohlc_quotes(1h)': '2h'}
2025-02-01 22:00:00.000 [🐞] (runner) [SimulationRunner] :: Running simulation from 2025-02-01 22:00:00 to 2025-02-02 04:00:00
2025-02-01 22:00:00.000 [🐞] (data)  | Warming up ('ohlc_quotes(1h)', BINANCE.UM:SWAP:BTCUSDT) -> 2h
2025-02-01 22:00:00.000 [ℹ️] (runner) SimulationRunner ::: Simulation started at 2025-02-01 22:00:00 :::


Simulating:   0%|          | 0/100 [00:00<?, ?%/s]

2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_warmup_finished
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 warmup finished completed
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_fit
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 is fitted
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 warmup finished completed
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_fit
2025-02-01 22:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 is fitted
2025-02-01 22:59:59.000 [ℹ️] (3475903177) __ALL__(fundamental)[2025-01-01 : 2025-02-01] : (288 x 5)
2025-02-01 23:30:00.000 [🐞] (processing) Performing daily delisting check (checking 1 days ahead)
2025-02-01 23:59:59.000 [ℹ️] (3475903177) __ALL__(fundamental)[2025-01-01 : 2025-02-01] : (288 x 5)
2025-02-02 00:59:59.000 [ℹ️] (3475903177) __ALL__(fundamental)[2025-01-01 : 2025

In [ ]:
rr.read("BTCUSDT", "ohlc(1h)", "2025-12-27 21:00", "2026-01-01 00:00").to_pd().tail(11)

## Update csv storage for test

In [67]:
fu = StorageRegistry.get("qdb::quantlab").get_reader("COINGECKO", "FUNDAMENTAL")
fd = fu.read(["BTC", "ETH", "BCH"], "fundamental", "2025-01-01", "2026-01-02")
fd.to_pd(True)

asset        metric         value
timestamp  symbol                                  
2025-01-01 BCH      BCH    market_cap  8.606387e+09
           BCH      BCH         price  4.342702e+02
           BCH      BCH  total_volume  2.052807e+08
           BTC      BTC    market_cap  1.851840e+12
           BTC      BTC         price  9.350786e+04
...                 ...           ...           ...
2026-01-01 BTC      BTC         price  8.752018e+04
           BTC      BTC  total_volume  3.724508e+10
           ETH      ETH    market_cap  3.580474e+11
           ETH      ETH         price  2.966774e+03
           ETH      ETH  total_volume  1.664149e+10

[3294 rows x 3 columns]

In [68]:
fd.to_pd(True).to_csv("../../tests/data/storages/multi/COINGECKO/FUNDAMENTAL/__ALL__.fundamental.csv.gz")

In [54]:
stor = StorageRegistry.get("csv::../../tests/data/storages/multi/")
rx = stor.get_reader("COINGECKO", "FUNDAMENTAL")
d_ids =  rx.get_data_id()
print(d_ids)
print(rx.get_data_types(d_ids[0]))

['__ALL__']
['fundamental']


In [57]:
stor.get_reader("COINGECKO", "FUNDAMENTAL").read(DataType.ALL, "fundamental", "2020-01-01", "now").to_pd(True)#.reset_index().set_index(["timestamp", "symbol"])

asset        metric         value
timestamp  symbol                                  
2023-01-01 BCH      BCH    market_cap  1.862681e+09
           BCH      BCH         price  9.694318e+01
           BCH      BCH  total_volume  3.309163e+08
           BTC      BTC    market_cap  3.182783e+11
           BTC      BTC         price  1.654069e+04
...                 ...           ...           ...
2023-12-31 BTC      BTC         price  4.222061e+04
           BTC      BTC  total_volume  1.472722e+10
           ETH      ETH    market_cap  2.757124e+11
           ETH      ETH         price  2.294344e+03
           ETH      ETH  total_volume  1.389624e+10

[3285 rows x 3 columns]

In [27]:
len(fu.get_data_id())

1268

In [20]:
xd = fd.to_pd(True)

In [ ]:
xd

In [ ]:
set(xd.index.get_level_values(1))

In [28]:
r1 = StorageRegistry.get("qdb::quantlab")["BINANCE.UM", "SWAP"]
r2 = StorageRegistry.get("qdb::quantlab")["KRAKEN", "SWAP"]
r3 = StorageRegistry.get("qdb::quantlab")["HYPERLIQUID", "SWAP"]

In [ ]:
r1.get_time_range("BTCUSDT", "quote")

In [ ]:
r2.get_time_range("BTCUSD", "quote")

In [ ]:
r3.get_time_range("BTCUSDC", "quote")

In [ ]:
if 0:
    q1 = r1.read("BTCUSDT", "quote", "2026-01-01", "2026-01-03")
    q2 = r2.read("BTCUSD", "quote", "2026-01-01", "2026-01-03")
    q3 = r3.read("BTCUSDC", "quote", "2026-01-01", "2026-01-03")

    q1.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/BINANCE.UM/SWAP/BTCUSDT.quote.csv.gz")
    q2.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/KRAKEN.F/SWAP/BTCUSD.quote.csv.gz")
    q3.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/HYPERLIQUID/SWAP/BTCUSDC.quote.csv.gz")

In [ ]:
if 0:
    sm = "BCH"
    q1 = r1.read(f"{sm}USDT", "ohlc(1h)", "2025-01-01", "2026-01-03")
    q2 = r2.read(f"{sm}USD", "ohlc(1h)", "2025-01-01", "2026-01-03")
    q3 = r3.read(f"{sm}USDC", "ohlc(1h)", "2025-01-01", "2026-01-03")

    q1.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/BINANCE.UM/SWAP/{sm}USDT.1H.csv.gz")
    q2.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/KRAKEN.F/SWAP/{sm}USD.1H.csv.gz")
    q3.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/HYPERLIQUID/SWAP/{sm}USDC.1H.csv.gz")

## Convert some data

In [8]:
from qubx.pandaz.utils import ohlc_resample

In [34]:
r0 = StorageRegistry.get("csv::../../tests/data/storages/csv").get_reader("BINANCE.UM", "SWAP")

In [ ]:
# ohlc_resample(r0.read("BTCUSDT", "ohlc(1min)").to_pd(), "5Min").to_csv(f"../../tests/data/storages/csv/BINANCE.UM/SWAP/BTCUSDT.5min.csv.gz")

In [ ]:
# for s in r0.get_data_id():
#     xi = r0.read(s, "ohlc(1h)", *r0.get_time_range(s, "ohlc(1h)")).to_pd()
#     ohlc_resample(xi, "1d").to_csv(f"../../tests/data/storages/csv/BINANCE.UM/SWAP/{s}.1D.csv.gz")


In [36]:
r0.read("BTCUSDT", "ohlc(5Min)").to_pd()

,open,high,low,close,volume
time,,,,,
2024-01-01 00:00:00,42314.0,42437.2,42289.6,42437.1,1724.210
2024-01-01 00:05:00,42437.2,42474.1,42420.5,42446.8,994.003
2024-01-01 00:10:00,42446.8,42535.0,42445.2,42532.5,899.775
2024-01-01 00:15:00,42532.4,42603.2,42494.1,42494.1,1291.232
2024-01-01 00:20:00,42494.1,42533.1,42484.4,42509.4,418.463
...,...,...,...,...,...
2024-02-15 05:55:00,51890.4,51985.6,51836.4,51982.9,1572.514
2024-02-15 06:00:00,51982.9,52008.2,51923.3,51943.4,789.807
2024-02-15 06:05:00,51943.4,51969.2,51860.9,51968.3,624.520
